# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all fields and record sets by their `@id` as required by the dataset schema.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary of the dataset
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Let's list all the record sets and for each record set, list its fields (all by `@id`).

In [ ]:
# List all RecordSet `@id` values and field `@id`s for exploratory purposes

record_sets = dataset.metadata.recordSets or []
print(f"Number of record sets: {len(record_sets)}")
recordset_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name if hasattr(rs, 'name') else ''} | @id: {rs.id}")
    recordset_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        field_ids = [f.id for f in rs.fields]
        print(f"  Field @id values: {field_ids}")
    if hasattr(rs, 'columns') and rs.columns:
        column_ids = [c.id for c in rs.columns]
        print(f"  Column @id values: {column_ids}")
    print()

## 3. Data Extraction

Let's extract the records from each record set using their `@id`, creating a DataFrame per record set for further analysis. We'll inspect the columns (`@id`) of the main clinical data set.

In [ ]:
dataframes = {}
# Use recordSets from the earlier cell (all by @id)
for rs_id in recordset_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for RecordSet @id '{rs_id}': shape={df.shape}")

# Pick the first recordset for further demonstration
main_recordset_id = recordset_ids[0] if recordset_ids else None
if main_recordset_id is not None:
    print(f"\nColumns in '{main_recordset_id}':")
    print(dataframes[main_recordset_id].columns.tolist())
    display(dataframes[main_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filter records, normalize numeric fields, and group/categorize data.

We'll identify a likely numeric field (e.g., age or interval in months between primary cancers). Please update field IDs as appropriate by inspecting previous outputs.

In [ ]:
# Inspect the columns to find a numeric field, such as interval_months, age, etc.
# For demonstration, we assume the numeric field has @id 'interval_months'
# and group by 'MSI_status' (replace with actual @id from above outputs as necessary)

# Replace with actual field @id from your schema if necessary
numeric_field_id = 'interval_months'  # <-- update if needed
group_field_id = 'MSI_status'        # <-- update if needed

df = dataframes[main_recordset_id]

if numeric_field_id in df.columns:
    # Remove missing or invalid entries
    valid_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
    valid_df[numeric_field_id] = pd.to_numeric(valid_df[numeric_field_id])

    threshold = 12
    filtered_df = valid_df[valid_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {filtered_df.shape[0]} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped {numeric_field_id} average by '{group_field_id}':")
        display(grouped_df.head())
else:
    print(f"Field '{numeric_field_id}' not found in columns: {df.columns.tolist()}")

## 5. Visualization

Visualize data distributions or relationships between fields. For example, plot the distribution of the interval between cancers, or compare mean intervals between MSI-H and MSI-L/MSS groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(valid_df[numeric_field_id], bins=16, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Compare by MSI_status if available
    if group_field_id in valid_df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(data=valid_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR² colorectal cancer survivor dataset using `mlcroissant`, referencing all entries by their `@id`. We loaded record sets, inspected their fields, and performed basic EDA such as filtering and normalizing a clinical interval, and grouping by MSI status. Visualizations highlighted distribution insights and group differences.

This framework can be extended for hypothesis testing or domain-specific machine learning. Always ensure references to entities use their `@id` as defined in the schema for robust, reproducible analysis.